# DAY - 29
# DATE - 19.06.2026
# Deep Learning Advanced — Transfer Learning for Satellite Imagery

In [20]:
import torch
import torch.nn as nn
import torchvision.models as models

In [21]:
# 1. Load pretrained ResNet18
# weights=ResNet18_Weights.DEFAULT replaces pretrained=True in newer torchvision versions
model = models.resnet18(weights = models.ResNet18_Weights.DEFAULT)

# 2. Extract original conv1 properties
original_conv = model.conv1
in_channels = 13     # # Changing from 3 to 13 for multispectral satellite data
out_channels = original_conv.out_channels
kernel_size = original_conv.kernel_size
stride = original_conv.stride
padding = original_conv.padding
bias = original_conv.bias

# 3. Create a new Conv2d layer
new_conv = nn.Conv2d(in_channels, out_channels, kernel_size = kernel_size, stride = stride,
                     padding = padding, bias = bias)

In [22]:
# 4. Strategy to handle weights: Copy RGB weights to first 3 channels,
# and initialize the rest (e.g., with the average of RGB weights)
with torch.no_grad():
     # Copy old weights into the first 3 channels
     new_conv.weight[:, :3, :, :] = original_conv.weight

     # Fill remaining 10 channels with the average of the RGB weights
     avg_weight = original_conv.weight.mean(dim = 1, keepdim = True)
     for i in range(3, 13):
         new_conv.weight[:, i:i+1, :, :] = avg_weight


# Replace the original conv1 with our new_conv
model.conv1 = new_conv

In [23]:
# 5. Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# 6. Modify and unfreeze the last fully connected layer (4 classes)
# ResNet18's final layer is named 'fc'
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 4)       # Aurtomatically Sets
requires_grad = True

print("Model modified successfully!")

Model modified successfully!


In [24]:
print(model)

ResNet(
  (conv1): Conv2d(13, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
 